In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/DomesticDeclarations.xes")

C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/10500 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56437 entries, 0 to 56436
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   id                      56437 non-null  object             
 1   org:resource            56437 non-null  object             
 2   concept:name            56437 non-null  object             
 3   time:timestamp          56437 non-null  datetime64[ns, UTC]
 4   org:role                56437 non-null  object             
 5   case:id                 56437 non-null  object             
 6   case:concept:name       56437 non-null  object             
 7   case:BudgetNumber       56437 non-null  object             
 8   case:DeclarationNumber  56437 non-null  object             
 9   case:Amount             56437 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(1), object(8)
memory usage: 4.3+ MB


In [6]:
df.isnull().any()

id                        False
org:resource              False
concept:name              False
time:timestamp            False
org:role                  False
case:id                   False
case:concept:name         False
case:BudgetNumber         False
case:DeclarationNumber    False
case:Amount               False
dtype: bool

In [7]:
df = df.drop(columns=['case:BudgetNumber', 'case:DeclarationNumber', 'case:id', 'id'])

In [8]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['org:resource'] = df['org:resource'].astype('string')
df['org:role'] = df['org:role'].astype('string')

df['case:Amount'] = df['case:Amount'].astype(np.float32)

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

In [9]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [10]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds().astype(np.float32)
df['time_delta'] = df['time_delta'].fillna(0)

In [11]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [39]:
# Remove activities only found in validation data
df = df[df['concept:name'] != 'Declaration FOR_APPROVAL by PRE_APPROVER']

In [40]:
df.head(20)

,case:concept:name,time:timestamp,case:Amount,concept:name,org:resource,org:role,time_delta
12788,declaration 100000,2018-01-30 09:20:07,600.844116,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
12789,declaration 100000,2018-02-07 09:58:46,600.844116,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,693519.0
12790,declaration 100000,2018-02-08 10:59:05,600.844116,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,90019.0
12791,declaration 100000,2018-02-09 12:42:49,600.844116,Request Payment,SYSTEM,UNDEFINED,92624.0
12792,declaration 100000,2018-02-12 17:31:20,600.844116,Payment Handled,SYSTEM,UNDEFINED,276511.0
12798,declaration 100005,2018-01-30 09:38:54,35.133686,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
12799,declaration 100005,2018-01-30 09:38:57,35.133686,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0
12800,declaration 100005,2018-01-30 10:04:10,35.133686,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,1513.0
12801,declaration 100005,2018-01-31 12:45:18,35.133686,Request Payment,SYSTEM,UNDEFINED,96068.0
12802,declaration 100005,2018-02-01 17:31:17,35.133686,Payment Handled,SYSTEM,UNDEFINED,103559.0


In [13]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 10500


### --- Feature Configurations ---

In [41]:
# --- Define feature specs ---
feature_specs = {

    "time_delta": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.20,
        "quantile_high":  0.80, 
    },

    "case:Amount": {
        "type":           "continuous",
        "level":          "case",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90,
    },

    "org:resource": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "org:role": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    # immutable
    "concept:name": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [42]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [43]:
# feature_config = FeatureConfig.load()

In [44]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Amount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [2.00, 284911.00]                        42289.5000 quantile_derived    
case:Amount                    continuous     case     yes    [6.91, 219.03]                           25.3885    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
org:role                       categorical    event    ye

### --- Next event prediction model ---

In [18]:
# Transform nan cols to NA
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [19]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [45]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [46]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [47]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['org:resource', 'org:role', 'concept:name']
  activity_prototypes: 16 activities


In [23]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [25]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=train_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=val_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 16, 'org:resource': 2, 'org:role': 7}, 'static_categorical_info': {}}


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
device

device(type='cuda')

In [30]:
criterion = torch.nn.CrossEntropyLoss()

In [31]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Dom-model_output.txt")

Epoch 020/100 | Train Loss: 0.3644 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.3570 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.3520 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.3488 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.3469 | LR: 1.00e-06
Time taken for next event model (training): 858.177677 seconds
Time taken for next event model (validation): 0.545610 seconds
Val loss: {'loss': 0.39033728538844215, 'accuracy': 0.8626302083333334, 'f1_macro': 0.5323280388885538, 'f1_weighted': 0.8297468184552721}


In [32]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [33]:
# model = ProcessLSTM.load()

In [34]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [38]:
# --- Save processed df ---
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/bpic20_Dom.xlsx", index=False, engine="openpyxl")

In [36]:
sys.stdout = original_stdout
log_file.close()